---
# Section 0: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
DATA_path= "/content/AmesHousing.csv"

df = pd.read_csv(DATA_path)

print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

---
# Section 1: Where We Left Off

The five problems you found last class. Confirm each one with a number.

In [ ]:
print('=' * 60)
print('PROBLEM 1: SKEW')
print('=' * 60)
print(f"SalePrice min    : {df['SalePrice'].min():>10,}")
print(f"SalePrice median : {df['SalePrice'].median():>10,.0f}")
print(f"SalePrice mean   : {df['SalePrice'].mean():>10,.0f}")
print(f"SalePrice max    : {df['SalePrice'].max():>10,}")
print()
print('The mean sits above the median. The tail pulled it up.')

In [ ]:
print('=' * 60)
print('PROBLEM 2: TEXT CATEGORIES A MODEL CANNOT READ')
print('=' * 60)
text_cols = df.select_dtypes(include='object').columns
print('Text columns:', len(text_cols))
print()
print('Neighborhood values:', df['Neighborhood'].nunique())
print(df['Neighborhood'].value_counts().head(3))
print('...')
print(df['Neighborhood'].value_counts().tail(3))

In [ ]:
print('=' * 60)
print('PROBLEM 3: WILDLY DIFFERENT SCALES')
print('=' * 60)
scale_cols = ['Lot Area', 'Gr Liv Area', 'Overall Qual', 'Overall Cond', 'Bedroom AbvGr']
print(df[scale_cols].agg(['min', 'max']).T)
print()
print('Lot Area range is tens of thousands. Overall Qual range is single digits.')

In [ ]:
print('=' * 60)
print('PROBLEM 4: OUTLIERS')
print('=' * 60)
print('Three largest lots:')
print(df.nlargest(3, 'Lot Area')[['Lot Area', 'MS Zoning', 'SalePrice', 'Neighborhood']])
print()
print('All zoned RL, residential low density. Real houses, not errors.')

In [ ]:
print('=' * 60)
print('PROBLEM 5: MISSING VALUES, TWO KINDS')
print('=' * 60)
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
summary = pd.DataFrame({'missing': miss, 'percent': (miss / len(df) * 100).round(1)})
print(summary)

Look at that table and split it in your head:

- **Near 100 percent blank** (`Pool QC`, `Misc Feature`, `Alley`, `Fence`):
  the feature does not exist. Not applicable.
- **Partly blank** (`Lot Frontage`, `Garage Yr Blt`, `Mas Vnr Area`):
  the data was lost. Not collected.

Same blank cell. Opposite treatment.

---
# Section 2: Data Cleaning, Missing Values

Three responses: eliminate, estimate, ignore.

## 2.1 What eliminating actually costs

Run this before you ever type `dropna()`.

In [ ]:
print(f"Rows now             : {len(df):>6,}")
print(f"Rows after dropna()  : {len(df.dropna()):>6,}")
lost = (1 - len(df.dropna()) / len(df)) * 100
print(f"Percent destroyed    : {lost:>6.1f} %")
print()
print('Dropping columns instead of rows:')
print(f"Columns now                   : {df.shape[1]:>4}")
print(f"Columns after dropna(axis=1)  : {df.dropna(axis=1).shape[1]:>4}")

## 2.2 Pool QC: the blank IS the information

99.7 percent blank. Do not impute. Do not drop.

In [ ]:
print('Pool QC value counts, blanks included:')
print(df['Pool QC'].value_counts(dropna=False))
print()
print('Cross-check against Pool Area:')
print('Houses with Pool Area == 0 :', (df['Pool Area'] == 0).sum())
print('Houses with Pool QC blank  :', df['Pool QC'].isna().sum())
print()
print('The numbers match. The blank means NO POOL. It is not lost data.')

In [ ]:
# The correct fix: make the blank a real category, then add a binary flag
work = df.copy()

NOT_APPLICABLE = ['Pool QC', 'Fence', 'Misc Feature', 'Alley', 'Fireplace Qu',
                  'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond',
                  'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
                  'BsmtFin Type 1', 'BsmtFin Type 2', 'Mas Vnr Type']

for c in NOT_APPLICABLE:
    if c in work.columns:
        work[c] = work[c].fillna('None')

work['HasPool']     = (work['Pool QC']     != 'None').astype(int)
work['HasFence']    = (work['Fence']       != 'None').astype(int)
work['HasFireplace']= (work['Fireplace Qu']!= 'None').astype(int)

print('Missing values remaining in those columns:', work[NOT_APPLICABLE].isna().sum().sum())
print()
print(work[['Pool QC', 'HasPool', 'Fence', 'HasFence']].head())
print()
print('HasPool counts:')
print(work['HasPool'].value_counts())

One useless column became one usable feature. Nothing was invented and nothing was lost.

## 2.3 Lot Frontage: the data really was lost

Here imputation is the right call. Three ways, worst to best.

In [ ]:
lf = df['Lot Frontage']
print(f"Missing        : {lf.isna().sum()} of {len(df)}  ({lf.isna().mean()*100:.1f} %)")
print(f"Overall median : {lf.median():.1f}")
print()
print('Median Lot Frontage by Neighborhood (first 8):')
print(df.groupby('Neighborhood')['Lot Frontage'].median().head(8))
print()
print('These differ a lot. A single global median ignores all of it.')

In [ ]:
# Option 1: overall median. Fast, ignores location.
opt1 = df['Lot Frontage'].fillna(df['Lot Frontage'].median())

# Option 2: median within the same Neighborhood. The right answer here.
opt2 = df.groupby('Neighborhood')['Lot Frontage'].transform(
           lambda s: s.fillna(s.median()))
opt2 = opt2.fillna(df['Lot Frontage'].median())   # safety net for tiny groups

compare = pd.DataFrame({
    'original'          : df['Lot Frontage'],
    'global_median'     : opt1,
    'neighborhood_median': opt2,
}).describe().T
print(compare[['mean', 'std', 'min', '50%', 'max']].round(2))
print()
print('Look at the std column. The global median shrinks the spread more,')
print('because it gives 490 houses the identical value.')

In [ ]:
# Option 3: KNN imputation. Use the most similar houses to guess.
from sklearn.impute import KNNImputer

knn_cols = ['Lot Frontage', 'Lot Area', 'Gr Liv Area', 'Overall Qual', 'Year Built']
imputer  = KNNImputer(n_neighbors=5)
knn_out  = imputer.fit_transform(df[knn_cols])
opt3     = pd.Series(knn_out[:, 0], index=df.index)

blank = df['Lot Frontage'].isna()
side_by_side = pd.DataFrame({
    'global_median'      : opt1[blank],
    'neighborhood_median': opt2[blank],
    'knn'                : opt3[blank],
}).head(10).round(1)
print('The 490 houses that were blank, filled three different ways:')
print(side_by_side)

In [ ]:
# Visual comparison of the three strategies
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for a, (name, s) in zip(ax, [('Global median', opt1),
                             ('Neighborhood median', opt2),
                             ('KNN, k=5', opt3)]):
    a.hist(s, bins=45, edgecolor='black')
    a.set_title(name)
    a.set_xlabel('Lot Frontage')
ax[0].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print('The global median produces a spike. It gave 490 houses one identical value.')

In [ ]:
work['Lot Frontage'] = opt2
print('Lot Frontage missing after imputation:', work['Lot Frontage'].isna().sum())

---
# Section 3: Data Cleaning, Duplicates, Outliers, Noise

## 3.1 Duplicates

Decide what "the same" means before you delete anything.

In [ ]:
print('Exact duplicate rows :', df.duplicated().sum())
print('Duplicate PID values :', df['PID'].duplicated().sum())
print('Duplicate Order      :', df['Order'].duplicated().sum())
print()
print('PID identifies a parcel. Order identifies a sale record.')
print('A repeated PID would mean the same property sold twice, which is NOT a duplicate.')

In [ ]:
# Now build a duplicate on purpose, so you can see the damage
dirty = pd.concat([df, df.head(50)], ignore_index=True)

print('Rows                    :', len(dirty))
print('Exact duplicates        :', dirty.duplicated().sum())
print()
print(f"Mean SalePrice, clean   : {df['SalePrice'].mean():>10,.0f}")
print(f"Mean SalePrice, dirty   : {dirty['SalePrice'].mean():>10,.0f}")
print(f"Count, clean            : {len(df):>10,}")
print(f"Count, dirty            : {len(dirty):>10,}")
print()
print('Fix:')
print('Rows after drop_duplicates():', len(dirty.drop_duplicates()))

## 3.2 Outliers

Real values, far from the rest. The record is correct.

In [ ]:
big = df.nlargest(5, 'Lot Area')[['Lot Area', 'MS Zoning', 'Neighborhood',
                                  'Gr Liv Area', 'Overall Qual', 'SalePrice']]
print('Five largest lots:')
print(big)
print()
print(f"Median Lot Area : {df['Lot Area'].median():>10,.0f}")
print(f"Largest Lot Area: {df['Lot Area'].max():>10,.0f}")
print(f"Ratio           : {df['Lot Area'].max() / df['Lot Area'].median():>10,.1f} x the median")

In [ ]:
# The scatter plot that reveals the famous Ames outliers
plt.figure(figsize=(8, 5.5))
plt.scatter(df['Gr Liv Area'], df['SalePrice'], alpha=0.35, s=16)
plt.axvline(4000, color='red', ls='--', label='4000 sq ft')
plt.xlabel('Gr Liv Area (sq ft)'); plt.ylabel('SalePrice ($)')
plt.title('The plot De Cock tells instructors to make')
plt.legend(); plt.tight_layout(); plt.show()

oversized = df[df['Gr Liv Area'] > 4000]
print(oversized[['Gr Liv Area', 'SalePrice', 'Overall Qual',
                 'Sale Condition', 'Yr Sold']])

**Read the `Sale Condition` column.**

De Cock's documentation says three of these are **Partial Sales**, which likely do not
represent actual market values. Two are simply unusual, very large houses priced correctly.

So: outlier or noise? Here you have a documented answer. Usually you will not.

In [ ]:
# Outliers are a SCALE problem, not a cleaning problem.
# Compare the boxplot before and after a log transform. Nothing was deleted.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

df.boxplot(column='Lot Area', ax=ax[0])
ax[0].set_title('Lot Area, raw')

pd.Series(np.log1p(df['Lot Area'])).plot.box(ax=ax[1])
ax[1].set_title('log(1 + Lot Area)')

plt.tight_layout(); plt.show()
print('Same 2,930 houses. The distribution is now readable.')

## 3.3 Noise

A wrong value. Find it with a rule, not with a plot.

In [ ]:
# RULE 1: range check. A year cannot be in the future.
print('Garage Yr Blt max :', df['Garage Yr Blt'].max())
print('Year Built max    :', df['Year Built'].max())
print('Yr Sold max       :', df['Yr Sold'].max())
print()
impossible = df[df['Garage Yr Blt'] > 2010]
print(f'Garage years after 2010: {len(impossible)}')
if len(impossible):
    print(impossible[['Garage Yr Blt', 'Year Built', 'Yr Sold', 'Garage Type']])

In [ ]:
# RULE 2: cross-column check. A garage cannot predate the house by decades.
both = df[['Year Built', 'Garage Yr Blt']].dropna()
odd  = both[both['Garage Yr Blt'] < both['Year Built']]
print(f'Garage built before the house: {len(odd)} rows')
print(odd.head(10))
print()
print('Some of these are legitimate. A house can be rebuilt on an old lot.')
print('Others are data entry errors. You cannot tell from the numbers alone.')

In [ ]:
# The fix for confirmed noise: treat it as missing, then use Section 2
work['Garage Yr Blt'] = work['Garage Yr Blt'].where(
    work['Garage Yr Blt'] <= 2010, np.nan)
print('Garage Yr Blt blanks after the range check:',
      work['Garage Yr Blt'].isna().sum())

---
# Section 4: Making Data Usable, Discretization

Turn a numeric column into a small number of ordered bins.

## 4.1 The equal width trap

Do this arithmetic out loud before you run the cell.

In [ ]:
lo, hi = df['SalePrice'].min(), df['SalePrice'].max()
rng_   = hi - lo
width  = rng_ / 4

print(f"SalePrice runs from {lo:,} to {hi:,}")
print(f"That is a range of {rng_:,}")
print(f"Cut into four equal bins. Each bin is {width:,.0f} wide.")
print()
edges = [lo + i * width for i in range(5)]
for i in range(4):
    print(f"  Bin {i+1}: {edges[i]:>10,.0f} to {edges[i+1]:>10,.0f}")
print()
print(f"Now, where do the houses actually live? The median is {df['SalePrice'].median():,.0f}")

In [ ]:
eq_width = pd.cut(df['SalePrice'], bins=4)
counts   = eq_width.value_counts().sort_index()

print('EQUAL WIDTH, houses per bin')
for interval, n in counts.items():
    print(f"  {str(interval):>28s} : {n:>5,}  ({n/len(df)*100:5.1f} %)")
print()
print('Four bins. One of them is the entire market. One of them is a handful of mansions.')
print('The cuts were mathematically perfect and completely useless.')

In [ ]:
# Equal frequency: same number of houses per bin. Widths differ, and that is fine.
eq_freq = pd.qcut(df['SalePrice'], q=4)
print('EQUAL FREQUENCY, houses per bin')
for interval, n in eq_freq.value_counts().sort_index().items():
    print(f"  {str(interval):>28s} : {n:>5,}  ({n/len(df)*100:5.1f} %)")

In [ ]:
# K-means: put the cuts where the data naturally clusters
from sklearn.preprocessing import KBinsDiscretizer

kb = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='kmeans', subsample=None)
km_bins = kb.fit_transform(df[['SalePrice']]).ravel().astype(int)

print('K-MEANS, houses per bin')
for b, n in pd.Series(km_bins).value_counts().sort_index().items():
    print(f"  bin {b} : {n:>5,}  ({n/len(df)*100:5.1f} %)")
print()
print('Cut points chosen by k-means:')
print([f'{e:,.0f}' for e in kb.bin_edges_[0]])

In [ ]:
# All three side by side
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
titles = ['Equal width', 'Equal frequency', 'K-means']
series = [eq_width.value_counts().sort_index().values,
          eq_freq.value_counts().sort_index().values,
          pd.Series(km_bins).value_counts().sort_index().values]

for a, t, s in zip(ax, titles, series):
    a.bar(range(1, 5), s, edgecolor='black')
    a.set_title(t); a.set_xlabel('bin'); a.set_xticks(range(1, 5))
ax[0].set_ylabel('houses')
plt.tight_layout(); plt.show()

print('Equal width divides the RANGE. The range is decided by your largest outlier.')

## 4.2 Year Built into four eras

138 distinct values become 4.

In [ ]:
print('Distinct values in Year Built:', df['Year Built'].nunique())
print('Range:', df['Year Built'].min(), 'to', df['Year Built'].max())

In [ ]:
work['Era'] = pd.cut(
    work['Year Built'],
    bins=[0, 1939, 1979, 1999, 2100],
    labels=['Pre-war', 'Post-war', 'Modern', 'New'])

print(work['Era'].value_counts().sort_index())
print()
print('Median SalePrice by era:')
print(work.groupby('Era', observed=True)['SalePrice'].median().round(0))
print()
print('138 values became 4, and now the result is a sentence a realtor understands.')

---
# Section 5: Making Data Usable, Binarization and Encoding

Every column must become a number before any algorithm can touch it.

In [ ]:
# One row, before and after
print('BEFORE')
print(work[['Foundation']].head(1))
print()
print('AFTER')
print(pd.get_dummies(work[['Foundation']].head(1)).T)
print()
print('Foundation values:', sorted(work['Foundation'].unique()))
print('One column becomes', work['Foundation'].nunique())

In [ ]:
# Binary column: the simplest case
work['Central Air'] = (df['Central Air'] == 'Y').astype(int)
print(work['Central Air'].value_counts())
print()
print('Y and N became 1 and 0. One column, done.')

## 5.1 Nominal versus ordinal: the step students get wrong

Both failure modes are **silent**. Nothing errors.

In [ ]:
QUALITY_MAP = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

ORDINAL_COLS = {
    'Kitchen Qual' : QUALITY_MAP,
    'Exter Qual'   : QUALITY_MAP,
    'Exter Cond'   : QUALITY_MAP,
    'Heating QC'   : QUALITY_MAP,
    'Bsmt Qual'    : QUALITY_MAP,
    'Bsmt Cond'    : QUALITY_MAP,
    'Fireplace Qu' : QUALITY_MAP,
    'Garage Qual'  : QUALITY_MAP,
    'Garage Cond'  : QUALITY_MAP,
    'Pool QC'      : QUALITY_MAP,
}

for col, mapping in ORDINAL_COLS.items():
    if col in work.columns:
        work[col + '_ord'] = work[col].map(mapping)

print(work[['Kitchen Qual', 'Kitchen Qual_ord']].drop_duplicates()
        .sort_values('Kitchen Qual_ord'))
print()
print('The ORDER is the information. Ex beats Gd beats TA. Keep it.')

---
# Section 6: Making Data Usable, Transformation and Scaling

## 6.1 The log transform, for skew

There are 2,930 houses. Most sit in a narrow band. A few stretch the axis.

In [ ]:
sp = df['SalePrice']
print(f"Houses between 100,000 and 300,000 : {sp.between(100_000, 300_000).sum():>5,}  "
      f"({sp.between(100_000, 300_000).mean()*100:4.1f} %)")
print(f"Houses above 500,000               : {(sp > 500_000).sum():>5,}  "
      f"({(sp > 500_000).mean()*100:4.1f} %)")
print()
print(f"Those few houses occupy "
      f"{(sp.max() - 500_000) / (sp.max() - sp.min()) * 100:.0f} % of the x axis.")
print()
print(f"mean   {sp.mean():>10,.0f}")
print(f"median {sp.median():>10,.0f}")
print(f"skew   {sp.skew():>10.2f}   (0 would be symmetric)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].hist(sp, bins=50, edgecolor='black')
ax[0].axvline(sp.mean(),   color='red',   lw=2, label=f'mean {sp.mean():,.0f}')
ax[0].axvline(sp.median(), color='green', lw=2, ls='--', label=f'median {sp.median():,.0f}')
ax[0].set_title(f'SalePrice, raw. skew = {sp.skew():.2f}')
ax[0].legend()

lsp = np.log(sp)
ax[1].hist(lsp, bins=50, edgecolor='black')
ax[1].axvline(lsp.mean(),   color='red',   lw=2, label='mean')
ax[1].axvline(lsp.median(), color='green', lw=2, ls='--', label='median')
ax[1].set_title(f'log(SalePrice). skew = {lsp.skew():.2f}')
ax[1].legend()

plt.tight_layout(); plt.show()

print('Log compressed the empty space. Mean and median now sit on top of each other.')

---
# Section 7: Making Data Smaller, Aggregation

## 7.1 Combining columns

Ames has four porch columns, all measured in square feet. Most are zero for most houses.

In [ ]:
PORCH = ['Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch']

print('Percent of houses with ZERO in each porch column:')
for c in PORCH:
    print(f'   {c:16s} {(df[c] == 0).mean()*100:5.1f} %')

work['Total Porch SF'] = df[PORCH].sum(axis=1)
work['Has Porch']      = (work['Total Porch SF'] > 0).astype(int)

print()
print(f"Percent with ZERO total porch : {(work['Total Porch SF'] == 0).mean()*100:5.1f} %")
print()
print('Correlation with SalePrice:')
for c in PORCH:
    print(f'   {c:16s} {df[c].corr(df["SalePrice"]):+.3f}')
print(f'   {"Total Porch SF":16s} {work["Total Porch SF"].corr(df["SalePrice"]):+.3f}')
print()
print('Four weak columns became one stronger one.')

In [ ]:
# Bathrooms: half baths count as half
BATH = ['Full Bath', 'Half Bath', 'Bsmt Full Bath', 'Bsmt Half Bath']
print(df[BATH].isna().sum())

work['Total Baths'] = (df['Full Bath'].fillna(0)
                       + 0.5 * df['Half Bath'].fillna(0)
                       + df['Bsmt Full Bath'].fillna(0)
                       + 0.5 * df['Bsmt Half Bath'].fillna(0))

print()
print(work['Total Baths'].value_counts().sort_index())
print()
print('Correlation with SalePrice:')
for c in BATH:
    print(f'   {c:16s} {df[c].corr(df["SalePrice"]):+.3f}')
print(f'   {"Total Baths":16s} {work["Total Baths"].corr(df["SalePrice"]):+.3f}')

## 7.2 Combining rows

2,930 sales become 28 neighborhoods.

In [ ]:
nbhd = df.groupby('Neighborhood').agg(
    houses      = ('SalePrice', 'size'),
    median_price= ('SalePrice', 'median'),
    median_area = ('Gr Liv Area', 'median'),
    mean_qual   = ('Overall Qual', 'mean'),
).sort_values('median_price', ascending=False).round(1)

print(f'{len(df):,} rows became {len(nbhd)} rows.')
print()
print(nbhd)

In [ ]:
# The stability effect: aggregated data varies less
print(f"Std dev of individual house prices  : {df['SalePrice'].std():>10,.0f}")
print(f"Std dev of neighborhood medians     : {nbhd['median_price'].std():>10,.0f}")
print()
ratio = df['SalePrice'].std() / nbhd['median_price'].std()
print(f'Individual prices vary {ratio:.1f} times more than neighborhood medians.')
print('Averaging cancelled the random variation. The signal survived.')

---
# Section 8: Making Data Smaller, Sampling

Ames does not need sampling. 2,930 rows fits in memory easily.
But the failure modes are visible even at this size.

In [ ]:
print('Rarest Neighborhood values:')
print(df['Neighborhood'].value_counts().tail(5))
print()
print('Houses with a pool:', (df['Pool Area'] > 0).sum(), 'out of', len(df))

In [ ]:
# Random 10 percent, repeated. Watch how many neighborhoods survive.
TOTAL_NB = df['Neighborhood'].nunique()
print(f'Full data has {TOTAL_NB} neighborhoods.')
print()
print('Random 10 percent samples:')
for seed in range(8):
    s = df.sample(frac=0.10, random_state=seed)
    print(f'   seed {seed}: {s["Neighborhood"].nunique():>2} neighborhoods, '
          f'{(s["Pool Area"] > 0).sum()} houses with a pool')

In [ ]:
# Stratified: sample within each neighborhood separately
strat = (df.groupby('Neighborhood', group_keys=False)
           .apply(lambda g: g.sample(frac=0.10, random_state=0), include_groups=False))

print('Stratified 10 percent sample')
print(f'   rows           : {len(strat):,}')
print(f'   neighborhoods  : {df.loc[strat.index, "Neighborhood"].nunique()} of {TOTAL_NB}')
print()
print('Every neighborhood present, in proportion. That is the point.')

In [ ]:
# With and without replacement
without = df.sample(n=500, replace=False, random_state=1)
with_   = df.sample(n=500, replace=True,  random_state=1)

print('WITHOUT replacement')
print('   unique rows drawn:', without.index.nunique(), 'of 500')
print('WITH replacement')
print('   unique rows drawn:', with_.index.nunique(), 'of 500')
print('   some rows appeared more than once')


---
# Section 9: Similarity and Distance

The realtor's question: **which houses are comparable to this one?**

In [ ]:
DIST_COLS = ['Lot Area', 'Overall Qual', 'Overall Cond', 'Gr Liv Area']

# The three houses from the slides
HOUSES = pd.DataFrame(
    [[5280, 4, 7, 1361],
     [8712, 4, 7,  952],
     [4800, 4, 5, 1844]],
    index=['A', 'B', 'C'], columns=DIST_COLS)

print(HOUSES)

## 9.1 Euclidean distance, step by step

Square the differences, add them, take the square root.

In [ ]:
import itertools

def euclidean_steps(frame, a, b):
    x, y = frame.loc[a].values.astype(float), frame.loc[b].values.astype(float)
    d, sq = x - y, (x - y) ** 2
    total = sq.sum()

    print(f'{a} to {b}')
    print('  ' + ''.join(f'{c:>16s}' for c in frame.columns))
    print('  differences ' + ''.join(f'{v:>16,.0f}' for v in d))
    print('  squared     ' + ''.join(f'{v:>16,.0f}' for v in sq))
    print(f'  sum = {total:,.0f}    sqrt = {np.sqrt(total):,.2f}')
    print('  share of the total distance:')
    for c, s in zip(frame.columns, sq):
        print(f'      {c:16s} {s/total*100:7.3f} %')
    print()
    return np.sqrt(total)

raw_euclid = {}
for a, b in itertools.combinations(HOUSES.index, 2):
    raw_euclid[(a, b)] = euclidean_steps(HOUSES, a, b)

In [ ]:
print('Nearest neighbour of A, using RAW data')
print(f"   A to B : {raw_euclid[('A','B')]:>10,.2f}")
print(f"   A to C : {raw_euclid[('A','C')]:>10,.2f}")
winner = 'B' if raw_euclid[('A','B')] < raw_euclid[('A','C')] else 'C'
print(f'   Nearest: {winner}')
print()
print('Look at what decided it. Lot Area contributed almost all of the distance.')
print('Overall Cond contributed essentially nothing.')

## 9.2 Manhattan distance

Same idea, no squaring.

In [ ]:
def manhattan_steps(frame, a, b):
    d = np.abs(frame.loc[a].values.astype(float) - frame.loc[b].values.astype(float))
    parts = ' + '.join(f'{v:,.0f}' for v in d)
    print(f'  {a} to {b}:  {parts}  =  {d.sum():,.0f}')
    return d.sum()

print('MANHATTAN, raw data')
raw_manh = {}
for a, b in itertools.combinations(HOUSES.index, 2):
    raw_manh[(a, b)] = manhattan_steps(HOUSES, a, b)
print()
print('Same ranking as Euclidean. Lot Area still dominates.')
print('Manhattan does not fix the scale problem.')

## 9.4 The scaling problem, and the answer flipping

Standardize every column, then compute the same distances again.

In [ ]:
# Fit the scaler on the WHOLE dataset, then apply it to our three houses.
# This is the correct order. Never fit on three rows.
scaler = StandardScaler().fit(df[DIST_COLS])

print('Column statistics from the full dataset:')
stats = pd.DataFrame({'mean': scaler.mean_,
                      'std' : np.sqrt(scaler.var_)},
                     index=DIST_COLS).round(2)
print(stats)
print()
print('USE THESE NUMBERS ON YOUR SLIDE, not my estimates.')

In [ ]:
HOUSES_Z = pd.DataFrame(scaler.transform(HOUSES),
                        index=HOUSES.index, columns=DIST_COLS)
print('STANDARDIZED VALUES (z-scores)')
print(HOUSES_Z.round(2))

In [ ]:
print('EUCLIDEAN ON STANDARDIZED DATA')
print()
z_euclid = {}
for a, b in itertools.combinations(HOUSES_Z.index, 2):
    x, y = HOUSES_Z.loc[a].values, HOUSES_Z.loc[b].values
    sq = (x - y) ** 2
    total = sq.sum()
    print(f'{a} to {b}')
    print('  ' + ''.join(f'{c:>16s}' for c in DIST_COLS))
    print('  differences ' + ''.join(f'{v:>16.2f}' for v in (x - y)))
    print('  squared     ' + ''.join(f'{v:>16.3f}' for v in sq))
    print(f'  sum = {total:.3f}    sqrt = {np.sqrt(total):.2f}')
    for c, s in zip(DIST_COLS, sq):
        print(f'      {c:16s} {s/total*100:6.2f} %')
    print()
    z_euclid[(a, b)] = np.sqrt(total)

In [ ]:
print('=' * 62)
print('THE PAYOFF')
print('=' * 62)
print(f"{'':22s}{'A to B':>12s}{'A to C':>12s}{'Nearest':>10s}")

rows = [
    ('Euclidean, raw',   raw_euclid[('A','B')], raw_euclid[('A','C')]),
    ('Manhattan, raw',   raw_manh[('A','B')],   raw_manh[('A','C')]),
    ('Euclidean, scaled', z_euclid[('A','B')],  z_euclid[('A','C')]),
]
for name, ab, ac in rows:
    print(f'{name:22s}{ab:>12,.2f}{ac:>12,.2f}{("B" if ab < ac else "C"):>10s}')

print()
print('Same houses. Same formula. Different answer.')
print('Scaling is not a formatting step. It decides your answer.')

## 9.5 Now do it on all 2,930 houses

Find the real comparables for one house, before and after scaling.

In [ ]:
from sklearn.metrics import pairwise_distances

# Pick a TYPICAL house, not an extreme one. An outlier's neighbours are
# other outliers, and then scaling changes nothing and the demo falls flat.
full   = df[DIST_COLS].dropna().reset_index(drop=True)
med    = full.median()
spread = full.std()
typicality = ((full - med).abs() / spread).sum(axis=1)
TARGET = int(typicality.idxmin())

print('Most typical house in the dataset is row', TARGET)
print()
full_z = pd.DataFrame(scaler.transform(full), index=full.index, columns=DIST_COLS)

d_raw = pairwise_distances(full.values,   full.values[[TARGET]], metric='euclidean').ravel()
d_z   = pairwise_distances(full_z.values, full_z.values[[TARGET]], metric='euclidean').ravel()

print('TARGET HOUSE')
print(full.iloc[TARGET])
print()

def top5(dist, label):
    order = np.argsort(dist)[1:6]        # skip position 0, the house itself
    out = full.iloc[order].copy()
    out['distance'] = dist[order].round(3)
    print(label)
    print(out)
    print()

top5(d_raw, 'FIVE NEAREST, RAW DATA')
top5(d_z,   'FIVE NEAREST, STANDARDIZED DATA')

overlap = set(np.argsort(d_raw)[1:6]) & set(np.argsort(d_z)[1:6])
print(f'Houses appearing in BOTH top-five lists: {len(overlap)} of 5')

## 9.6 Text columns still need a distance

`Foundation` is nominal. `Kitchen Qual` is ordinal. Neither is a number yet.

In [ ]:
def nominal_distance(v1, v2):
    return 0 if v1 == v2 else 1

def ordinal_distance(v1, v2, mapping):
    lo, hi = min(mapping.values()), max(mapping.values())
    return abs(mapping[v1] - mapping[v2]) / (hi - lo)

pairs = [('PConc', 'PConc'), ('PConc', 'CBlock'), ('PConc', 'Slab')]
print('NOMINAL: Foundation')
for a, b in pairs:
    print(f'   {a:8s} vs {b:8s} -> {nominal_distance(a, b)}')
print('   You cannot say PConc is nearer to CBlock than to Slab. There is no order.')
print()

print('ORDINAL: Kitchen Qual, mapped Po=1 Fa=2 TA=3 Gd=4 Ex=5')
KQ = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
for a, b in [('Gd', 'Fa'), ('Ex', 'Po'), ('TA', 'TA'), ('Ex', 'Gd')]:
    print(f'   {a:3s} vs {b:3s} -> {ordinal_distance(a, b, KQ):.2f}')
print()
print('Honest caveat: this assumes the steps are equally sized.')


---
# Section 10: The Curse of Dimensionality

In [ ]:

text_cols = df.select_dtypes(include='object').columns
print(f'Text columns encoded : {len(text_cols)}')
print()
enc = pd.get_dummies(df)
print(f'Columns before : {df.shape[1]}')
print(f'Columns after  : {enc.shape[1]}')
print(f'Rows           : {len(df):,}  (unchanged)')
print()
print(f'Rows per column, before : {len(df)/df.shape[1]:.1f}')
print(f'Rows per column, after  : {len(df)/enc.shape[1]:.1f}')

In [ ]:
# DISTANCE CONCENTRATION: the demo Tan uses.
# 500 random points. Watch the gap between nearest and farthest collapse.
rng = np.random.default_rng(0)
dims = [2, 5, 10, 20, 50, 100, 200, 500, 1000]
rows = []

for d in dims:
    pts = rng.random((500, d))
    D = pairwise_distances(pts)
    off = D[np.triu_indices(500, k=1)]
    rows.append({'dimensions': d,
                 'min': off.min(),
                 'max': off.max(),
                 'mean': off.mean(),
                 'relative_gap': (off.max() - off.min()) / off.mean()})

conc = pd.DataFrame(rows)
print(conc.round(4).to_string(index=False))
print()
print('Read the last column. That is the spread between the closest and')
print('farthest pair, relative to the average distance.')
print('It collapses toward zero as dimensions grow.')